Copyright 2026 Google LLC.

SPDX-License-Identifier: Apache-2.0

In [ ]:
# @title TIPSv2 Zero-Shot Segmentation DPT Notebook

#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at

#  https://www.apache.org/licenses/LICENSE-2.0

#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.

In [ ]:
# @title Install dependencies and clone TIPS repo
import os
import sys

# Root directory for all files (Colab default is /content).
ROOT_DIR = os.getcwd()
TIPS_DIR = os.path.join(ROOT_DIR, 'tips')

# Install required packages.
!pip install -q torch torchvision torchaudio
!pip install -q tensorflow_text mediapy jax jaxlib scikit-learn

# Clone the TIPS repository (branch with pytorch/decoders.py).
if not os.path.exists(TIPS_DIR):
  !git clone --branch add-decoders-module https://github.com/google-deepmind/tips.git {TIPS_DIR}

# Add the root directory to PYTHONPATH so that `tips.*` imports work.
if ROOT_DIR not in sys.path:
  sys.path.insert(0, ROOT_DIR)

print(f'ROOT_DIR: {ROOT_DIR}')
print(f'TIPS_DIR: {TIPS_DIR}')
print('Installation complete!')

In [ ]:
# @title Download the checkpoints and ADE20k Dataset
import urllib.request
import os
import zipfile

variant = 'L'  # @param ["B", "L", "So", "g"]

CHECKPOINT_BASE_URL = 'https://storage.googleapis.com/tips_data/v2_0/checkpoints/pytorch'
TOKENIZER_URL = 'https://storage.googleapis.com/tips_data/v1_0/checkpoints/tokenizer.model'
ADE20K_URL = 'http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip'
ADE20K_TMP_PATH = os.path.join(ROOT_DIR, 'ADEChallengeData2016.zip')
DPT_BASE_URL = 'https://storage.googleapis.com/tips_data/v2_0/checkpoints/pytorch'

CKPT_DIR = os.path.join(ROOT_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

V2_CHECKPOINT_MAP = {
    'B': ('tips_v2_oss_b14_vision.npz', 'tips_v2_oss_b14_text.npz'),
    'L': ('tips_v2_oss_l14_vision.npz', 'tips_v2_oss_l14_text.npz'),
    'So': ('tips_v2_oss_so14_vision.npz', 'tips_v2_oss_so14_text.npz'),
    'g': ('tips_v2_oss_g14_vision.npz', 'tips_v2_oss_g14_text.npz'),
}
vision_ckpt_name, text_ckpt_name = V2_CHECKPOINT_MAP[variant]

DPT_CHECKPOINT_MAP = {
    'B': 'tips_v2_b14_segmentation_dpt_pytorch.zip',
    'L': 'tips_v2_l14_segmentation_dpt_pytorch.zip',
    'So': 'tips_v2_so400m14_segmentation_dpt_pytorch.zip',
    'g': 'tips_v2_g14_segmentation_dpt_pytorch.zip',
}
dpt_ckpt_name = DPT_CHECKPOINT_MAP[variant]

for ckpt_name in [vision_ckpt_name, text_ckpt_name]:
  ckpt_path = os.path.join(CKPT_DIR, ckpt_name)
  if not os.path.exists(ckpt_path):
    print(f'\nDownloading {ckpt_name}...')
    urllib.request.urlretrieve(f'{CHECKPOINT_BASE_URL}/{ckpt_name}', ckpt_path)
    print(f'  Saved to {ckpt_path}')
  else:
    print(f'  {ckpt_name} already exists.')

dpt_ckpt_path = os.path.join(CKPT_DIR, dpt_ckpt_name)
if not os.path.exists(dpt_ckpt_path):
  print(f'\nDownloading DPT checkpoint {dpt_ckpt_name}...')
  urllib.request.urlretrieve(f'{DPT_BASE_URL}/{dpt_ckpt_name}', dpt_ckpt_path)
  print(f'  Saved to {dpt_ckpt_path}')
else:
  print(f'  {dpt_ckpt_name} already exists.')

tokenizer_file = os.path.join(CKPT_DIR, 'tokenizer.model')
if not os.path.exists(tokenizer_file):
  print('\nDownloading tokenizer...')
  urllib.request.urlretrieve(TOKENIZER_URL, tokenizer_file)
  print(f'  Saved to {tokenizer_file}')
else:
  print('  tokenizer.model already exists.')

# Download and extract ADE20K dataset for sample images.
ADE20K_IMG_DIR = os.path.join(ROOT_DIR, 'ADEChallengeData2016', 'images', 'validation')
if not os.path.isdir(ADE20K_IMG_DIR):
  print('\nDownloading ADEChallengeData2016.zip...')
  urllib.request.urlretrieve(ADE20K_URL, ADE20K_TMP_PATH)
  print('Extracting...')
  with zipfile.ZipFile(ADE20K_TMP_PATH, 'r') as z:
    z.extractall(ROOT_DIR)
  os.remove(ADE20K_TMP_PATH)
  print(f'  Extracted to {ADE20K_IMG_DIR}')
else:
  print('  ADE20K images already extracted.')

IMG_DIR = ADE20K_IMG_DIR
print('\nAll downloads complete!')

In [ ]:
#@title Configure the TIPS model.
# Set the input image shape and variant.
image_size = 448  # @param {type: "number"}
device = 'cuda'
# variant is already set in the download cell above.

# Checkpoint and tokenizer paths (absolute paths).
image_encoder_checkpoint = os.path.join(CKPT_DIR, vision_ckpt_name)
text_encoder_checkpoint = os.path.join(CKPT_DIR, text_ckpt_name)
tokenizer_path = os.path.join(CKPT_DIR, 'tokenizer.model')
dpt_checkpoint = os.path.join(CKPT_DIR, dpt_ckpt_name)

print(f'Image encoder checkpoint: {image_encoder_checkpoint}')
print(f'Text encoder checkpoint: {text_encoder_checkpoint}')
print(f'Tokenizer path: {tokenizer_path}')
print(f'DPT checkpoint: {dpt_checkpoint}')

In [ ]:
# @title Import DPT Segmentation Decoder from tips.pytorch.decoders

from tips.pytorch.decoders import SegmentationDecoder, load_decoder_weights

print('SegmentationDecoder and load_decoder_weights imported successfully!')

In [ ]:
# @title Load TIPS Model
import numpy as np
import torch
from tips.pytorch import image_encoder

PATCH_SIZE = 14
weights_image = dict(np.load(image_encoder_checkpoint, allow_pickle=False))
for key in weights_image:
    weights_image[key] = torch.tensor(weights_image[key])
ffn_layer = 'swiglu' if variant == 'g' else 'mlp'

MODEL_CONSTRUCTOR_MAP = {
    'B': 'vit_base',
    'L': 'vit_large',
    'So': 'vit_so400m',
    'g': 'vit_giant2',
}
EMBED_DIM_MAP = {
    'B': 768,
    'L': 1024,
    'So': 1152,
    'g': 1536,
}
INTERMEDIATE_LAYERS_MAP = {
    'B': [2, 5, 8, 11],
    'L': [5, 11, 17, 23],
    'So': [6, 13, 20, 26],
    'g': [9, 19, 29, 39],
}

vit_constructor = getattr(image_encoder, MODEL_CONSTRUCTOR_MAP[variant])
embed_dim = EMBED_DIM_MAP[variant]
intermediate_layers = INTERMEDIATE_LAYERS_MAP[variant]
post_process_channels = (embed_dim // 8, embed_dim // 4, embed_dim // 2, embed_dim)

with torch.no_grad():
  model_image = vit_constructor(
      img_size=image_size, patch_size=PATCH_SIZE, ffn_layer=ffn_layer,
      block_chunks=0, init_values=1.0,
      interpolate_antialias=True, interpolate_offset=0.0,
  )
  model_image.load_state_dict(weights_image)
  model_image.eval()

with torch.no_grad():
  dpt_model = SegmentationDecoder(
      num_classes=150,
      input_embed_dim=embed_dim,
      post_process_channels=post_process_channels,
  )
  load_decoder_weights(dpt_model, dpt_checkpoint)
  dpt_model.eval()

print('TIPS model and DPT head loaded successfully!')
print(f'  Variant: {variant}, embed_dim: {embed_dim}')
print(f'  Intermediate layers: {intermediate_layers}')

In [ ]:
# @title Define ADE20K Classes and Palette
import numpy as np
import colorsys

ADE20K_CLASSES = (
    'wall', 'building', 'sky', 'floor', 'tree',
    'ceiling', 'road', 'bed', 'windowpane', 'grass',
    'cabinet', 'sidewalk', 'person', 'earth', 'door',
    'table', 'mountain', 'plant', 'curtain', 'chair',
    'car', 'water', 'painting', 'sofa', 'shelf',
    'house', 'sea', 'mirror', 'rug', 'field',
    'armchair', 'seat', 'fence', 'desk', 'rock',
    'wardrobe', 'lamp', 'bathtub', 'railing', 'cushion',
    'base', 'box', 'column', 'signboard', 'chest_of_drawers',
    'counter', 'sand', 'sink', 'skyscraper', 'fireplace',
    'refrigerator', 'grandstand', 'path', 'stairs', 'runway',
    'case', 'pool_table', 'pillow', 'screen_door', 'stairway',
    'river', 'bridge', 'bookcase', 'blind', 'coffee_table',
    'toilet', 'flower', 'book', 'hill', 'bench',
    'countertop', 'stove', 'palm', 'kitchen_island', 'computer',
    'swivel_chair', 'boat', 'bar', 'arcade_machine', 'hovel',
    'bus', 'towel', 'light', 'truck', 'tower',
    'chandelier', 'awning', 'streetlight', 'booth', 'television',
    'airplane', 'dirt_track', 'apparel', 'pole', 'land',
    'bannister', 'escalator', 'ottoman', 'bottle', 'buffet',
    'poster', 'stage', 'van', 'ship', 'fountain',
    'conveyer_belt', 'canopy', 'washer', 'plaything',
    'swimming_pool',
    'stool', 'barrel', 'basket', 'waterfall', 'tent',
    'bag', 'minibike', 'cradle', 'oven', 'ball',
    'food', 'step', 'tank', 'trade_name', 'microwave',
    'pot', 'animal', 'bicycle', 'lake', 'dishwasher',
    'screen', 'blanket', 'sculpture', 'hood', 'sconce',
    'vase', 'traffic_light', 'tray', 'ashcan', 'fan',
    'pier', 'crt_screen', 'plate', 'monitor',
    'bulletin_board',
    'shower', 'radiator', 'glass', 'clock', 'flag',
)

NUM_ADE20K_CLASSES = 150
ADE20K_PALETTE = np.zeros((NUM_ADE20K_CLASSES + 1, 3), dtype=np.uint8)
for i in range(1, NUM_ADE20K_CLASSES + 1):
    hue = (i * 0.618033988749895) % 1.0
    saturation = 0.65 + 0.35 * ((i * 7) % 5) / 4.0
    value = 0.70 + 0.30 * ((i * 11) % 3) / 2.0
    r, g, b = colorsys.hsv_to_rgb(hue, saturation, value)
    ADE20K_PALETTE[i] = [int(r * 255), int(g * 255), int(b * 255)]

print(f"Defined {len(ADE20K_CLASSES)} ADE20K classes and palette.")

In [ ]:
# @title Run Inference on a Sample Image
import PIL.Image
import matplotlib.pyplot as plt
import torchvision.transforms as TVT
import os
import glob

# Load a sample ADE20K validation image.
ade20k_images = sorted(glob.glob(os.path.join(IMG_DIR, '*.jpg')))
image_path = ade20k_images[0]
print(f'Using image: {image_path}')
img = PIL.Image.open(image_path)

transform = TVT.Compose([TVT.Resize((image_size, image_size)), TVT.ToTensor()])
tensor = transform(img).unsqueeze(0)

with torch.no_grad():
    # get_intermediate_layers returns List[(feat, cls_token)] when return_class_token=True.
    features = model_image.get_intermediate_layers(
        tensor, n=intermediate_layers, reshape=True, return_class_token=True, norm=True
    )
    # ReassembleBlocks expects List[(cls_token, feat)] - swap the order.
    features = [(cls, feat) for feat, cls in features]
    # SegmentationDecoder returns (B, num_classes, H, W) - channel-first.
    seg_logits = dpt_model(features, image_size=(image_size, image_size))
    seg_map = seg_logits.argmax(dim=1).squeeze(0).cpu().numpy()

colored_map = ADE20K_PALETTE[seg_map]
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(img.resize((image_size, image_size))); plt.title("Input Image"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(colored_map); plt.title("Segmentation Map"); plt.axis("off")
plt.show()